# DBT Query Compiler

This notebook helps you compile and view dbt models locally to see what the compiled SQL looks like.

## 1. Import Required Libraries

In [1]:
import os
import subprocess
from pathlib import Path
from IPython.display import display, Markdown
import sqlparse

# Set the working directory to the transformation folder
os.chdir('/Users/tylerclifton/Projects/mbta-reliability-analytics/transform')
print(f"Current directory: {os.getcwd()}")

Current directory: /Users/tylerclifton/Projects/mbta-reliability-analytics/transform


## 2. Compile All Models

Run `dbt compile` to compile all models and macros in the project.

In [14]:
# Run dbt compile
# Using ! to run shell command in the notebook's kernel environment
!dbt compile

19:29:20  Running with dbt=1.7.8
19:29:20  Registered adapter: bigquery=1.7.4
19:29:20  Found 1 operation, 6 models, 0 sources, 0 exposures, 0 metrics, 467 macros, 0 groups, 0 semantic models
19:29:20  
19:29:21  Concurrency: 4 threads (target='dev')
19:29:21  
19:29:22  Encountered an error:
Runtime Error
  Runtime Error in operation mbta_reliability_analytics-on-run-end-0 (./dbt_project.yml)
    404 Not found: Dataset mbta-reliability-analytics:mbta_dev was not found in location us-east1; reason: notFound, message: Not found: Dataset mbta-reliability-analytics:mbta_dev was not found in location us-east1
    
    Location: us-east1
    Job ID: d4b69dac-8db6-477a-9395-8d3e8ec841ef
    


## 3. View Compiled SQL

Helper function to read and display compiled SQL from the target directory.

In [3]:
def view_compiled_sql(model_path):
    """
    View the EXACT compiled SQL for a given model - no formatting, just the raw file.
    
    Args:
        model_path: Path to the model relative to models/ (e.g., 'mbta/bronze/mbta_bronze_alerts.sql')
    """
    compiled_path = f"target/compiled/mbta_reliability_analytics/models/{model_path}"
    
    if not os.path.exists(compiled_path):
        print(f"❌ Compiled file not found: {compiled_path}")
        return
    
    # Read the actual compiled SQL exactly as-is
    with open(compiled_path, 'r') as f:
        sql_content = f.read()
    
    print(f"📄 Compiled SQL for: {model_path}")
    print("=" * 80)
    display(Markdown(f"```sql\n{sql_content}\n```"))
    print("=" * 80)
    
# Test the function
print("Function defined. Use it like: view_compiled_sql('mbta/bronze/mbta_bronze_alerts.sql')")

Function defined. Use it like: view_compiled_sql('mbta/bronze/mbta_bronze_alerts.sql')


## 4. View MBTA Bronze Alerts Model

In [4]:
view_compiled_sql('mbta/bronze/mbta_bronze_alerts.sql')

📄 Compiled SQL for: mbta/bronze/mbta_bronze_alerts.sql


```sql


SELECT
    active_period_start,
    active_period_end,
    alert_id,
    route,
    header,
    description,
    cause,
    effect,
    severity,
    duration_certainty,
    created_at,
    updated_at,
    ingestion_timestamp,
    ingestion_source
FROM `mbta-reliability-analytics.staging.`


```

## 5. View MBTA Bronze Routes Model

In [5]:
view_compiled_sql('mbta/bronze/mbta_bronze_routes.sql')

📄 Compiled SQL for: mbta/bronze/mbta_bronze_routes.sql


```sql


SELECT
    route_id,
    long_name,
    description,
    route_type,
    color,
    direction_destinations,
    ingestion_timestamp,
    ingestion_source
FROM `mbta-reliability-analytics.staging.`


```

## 6. View MBTA Silver Model

In [11]:
view_compiled_sql('mbta/silver/mbta_silver.sql')

📄 Compiled SQL for: mbta/silver/mbta_silver.sql


```sql


WITH base AS (
    SELECT
        CAST(REGEXP_EXTRACT(CAST(active_period_start AS STRING), r"^\d{4}-\d{2}-\d{2}") AS DATE) AS alert_start_date,
        CAST(REGEXP_EXTRACT(CAST(active_period_end AS STRING), r"^\d{4}-\d{2}-\d{2}") AS DATE) AS alert_end_date,
        CAST(alert_id AS STRING) AS alert_id,
        CAST(route AS STRING) AS route_id,
        CAST(header AS STRING) AS alert_header,
        CAST(description AS STRING) AS alert_description,
        CAST(cause AS STRING) AS alert_cause,
        CAST(effect AS STRING) AS alert_effect,
        CAST(severity AS INT64) AS alert_severity,
        CAST(duration_certainty AS STRING) AS alert_duration_certainty,
        CAST(REGEXP_EXTRACT(CAST(created_at AS STRING), r"^\d{4}-\d{2}-\d{2}") AS DATE) AS alert_created_at,
        CAST(REGEXP_EXTRACT(CAST(updated_at AS STRING), r"^\d{4}-\d{2}-\d{2}") AS DATE) AS alert_updated_at,
        CAST(ingestion_timestamp AS TIMESTAMP) AS ingestion_timestamp,
        CAST(ingestion_source AS STRING) AS ingestion_source
    FROM `mbta-reliability-analytics`.`mbta_dev`.`mbta_bronze_alerts`
)
,
routes_base AS (
    SELECT
        CAST(route_id AS STRING) AS route_id,
        CAST(long_name AS STRING) AS route_name,
        CAST(description AS STRING) AS route_description,
        CAST(route_type AS STRING) AS route_type,
        CAST(color AS STRING) AS route_color,
        CAST(direction_destinations AS STRING) AS route_destinations,
        CAST(ingestion_timestamp AS TIMESTAMP) AS ingestion_timestamp,
        CAST(ingestion_source AS STRING) AS ingestion_source
    FROM `mbta-reliability-analytics`.`mbta_dev`.`mbta_bronze_routes`
)

SELECT
    base.*
    ,
    r.route_id
    ,
    r.route_name
    ,
    r.route_description
    ,
    r.route_type
    ,
    r.route_color
    ,
    r.route_destinations
    ,
    r.ingestion_timestamp
    ,
    r.ingestion_source
FROM base
LEFT JOIN routes_base AS r
    ON base.route_id = r.route_id
```

## 7. View NWS Bronze Weather Model

In [7]:
view_compiled_sql('nws/bronze/nws_bronze_weather.sql')

📄 Compiled SQL for: nws/bronze/nws_bronze_weather.sql


```sql


SELECT
    observation_timestamp,
    station_id,
    temperature_fahrenheit,
    temperature_celsius,
    dewpoint_fahrenheit,
    dewpoint_celsius,
    wind_speed_mph,
    wind_direction_degrees,
    precipitation_last_hour_mm,
    relative_humidity_percent,
    barometric_pressure_pa,
    visibility_miles,
    cloud_base_meters,
    cloud_coverage,
    conditions,
    ingestion_timestamp,
    ingestion_source
FROM `mbta-reliability-analytics.staging.`


```

## 8. View NWS Silver Weather Model

In [12]:
view_compiled_sql('nws/silver/nws_silver.sql')

📄 Compiled SQL for: nws/silver/nws_silver.sql


```sql


WITH base AS (
    SELECT
        CAST(observation_timestamp AS TIMESTAMP) AS observation_timestamp,
        CAST(station_id AS STRING) AS station_id,
        CAST(temperature_fahrenheit AS FLOAT64) AS temperature_fahrenheit,
        CAST(temperature_celsius AS FLOAT64) AS temperature_celsius,
        CAST(dewpoint_fahrenheit AS FLOAT64) AS dewpoint_fahrenheit,
        CAST(dewpoint_celsius AS FLOAT64) AS dewpoint_celsius,
        CAST(wind_speed_mph AS FLOAT64) AS wind_speed_mph,
        CAST(wind_direction_degrees AS FLOAT64) AS wind_direction_degrees,
        CAST(precipitation_last_hour_mm AS FLOAT64) AS precipitation_last_hour_mm,
        CAST(relative_humidity_percent AS FLOAT64) AS relative_humidity_percent,
        CAST(barometric_pressure_pa AS FLOAT64) AS barometric_pressure_pa,
        CAST(visibility_miles AS FLOAT64) AS visibility_miles,
        CAST(cloud_base_meters AS FLOAT64) AS cloud_base_meters,
        CAST(cloud_coverage AS STRING) AS cloud_coverage,
        CAST(conditions AS STRING) AS conditions,
        CAST(ingestion_timestamp AS TIMESTAMP) AS ingestion_timestamp,
        CAST(ingestion_source AS STRING) AS ingestion_source
    FROM `mbta-reliability-analytics`.`mbta_dev`.`nws_bronze_weather`
)

SELECT
    base.*
FROM base
```

## 9. View Gold Layer: MBTA Alerts with Weather

In [15]:
view_compiled_sql('gold/mbta_alerts_with_weather.sql')

📄 Compiled SQL for: gold/mbta_alerts_with_weather.sql


```sql




WITH base AS (
    SELECT *
    FROM `mbta-reliability-analytics`.`mbta_dev`.`mbta_silver`
),
weather_agg AS (
    SELECT
        DATE(observation_timestamp) AS observation_date,
        AVG(temperature_fahrenheit) AS avg_temperature_f,
        MIN(temperature_fahrenheit) AS min_temperature_f,
        MAX(temperature_fahrenheit) AS max_temperature_f,
        AVG(precipitation_last_hour_mm) AS avg_precipitation_mm,
        MAX(precipitation_last_hour_mm) AS max_precipitation_mm,
        AVG(wind_speed_mph) AS avg_wind_speed_mph,
        MAX(wind_speed_mph) AS max_wind_speed_mph,
        AVG(visibility_miles) AS avg_visibility_miles,
        MIN(visibility_miles) AS min_visibility_miles,
        AVG(relative_humidity_percent) AS avg_humidity_percent
    FROM `mbta-reliability-analytics`.`mbta_dev`.`nws_silver`
    GROUP BY DATE(observation_timestamp)
)

SELECT
    base.*,
    weather_agg.avg_temperature_f,
    weather_agg.min_temperature_f,
    weather_agg.max_temperature_f,
    weather_agg.avg_precipitation_mm,
    weather_agg.max_precipitation_mm,
    weather_agg.avg_wind_speed_mph,
    weather_agg.max_wind_speed_mph,
    weather_agg.avg_visibility_miles,
    weather_agg.min_visibility_miles,
    weather_agg.avg_humidity_percent
FROM base
LEFT JOIN weather_agg
    ON base.alert_start_date = weather_agg.observation_date
```

## 10. List All Available Compiled Models

Use this to see all compiled SQL files in your target directory.

In [16]:
import glob

# Find all compiled SQL files
compiled_dir = "target/compiled/mbta_reliability_analytics/models/"
if os.path.exists(compiled_dir):
    sql_files = glob.glob(f"{compiled_dir}**/*.sql", recursive=True)
    
    print(f"Found {len(sql_files)} compiled SQL files:\n")
    for file_path in sorted(sql_files):
        # Get relative path from models directory
        rel_path = file_path.replace(compiled_dir, "")
        print(f"  • {rel_path}")
else:
    print(f"Compiled directory not found: {compiled_dir}")
    print("Run dbt compile first!")

Found 6 compiled SQL files:

  • gold/mbta_alerts_with_weather.sql
  • mbta/bronze/mbta_bronze_alerts.sql
  • mbta/bronze/mbta_bronze_routes.sql
  • mbta/silver/mbta_silver.sql
  • nws/bronze/nws_bronze_weather.sql
  • nws/silver/nws_silver.sql


## 11. View Any Custom Model

Use the function below to view any compiled model by specifying its path.

In [10]:
# Example: View any model by specifying its path
# view_compiled_sql('path/to/your/model.sql')

# Uncomment and modify the line below to view a specific model:
# view_compiled_sql('mbta/bronze/mbta_bronze_alerts.sql')